In [ ]:
# KD-ReID on Kaggle: train one or more configs in the background, then evaluate. Keep this as the ONLY cell,
# then "Save Version > Save & Run All". Settings: Accelerator = GPU T4 x2 (one GPU is used), Internet = On,
# Add-ons > Secrets: GH_TOKEN. Input dataset with: Market-1501 zip, CUHK03-NP zip, and the weights the
# configs need (market_vit_clipreid.pth for a CLIP teacher, osnet_x1_0_imagenet.pth / osnet_x0_25_imagenet.pth
# for OSNet students; ResNet ImageNet weights are downloaded by torchvision).
# RUNS: (config path under configs/, seed). Runs one after another in this session (Kaggle limit: 12 h).
# Output folder: /kaggle/working/outputs/<config dir>/<config name>[_s<seed>]  (seed 42 -> no suffix).
RUNS = [
    ("plan8/8a_unl_market", 1),
]
EXTRA = ""  # extra key=value overrides for every run, e.g. "train.eval_period=40"

import os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

IN, TMP = Path("/kaggle/input"), Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")
RAW, DATA, WORK = TMP / "raw", TMP / "data", Path("/kaggle/working")
REPO = TMP / "KD-ReID"

def sh(cmd):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode

# ===== 0. GPU + code =====
import torch
assert torch.cuda.is_available(), "Settings > Accelerator > GPU T4 x2"
print(torch.__version__, torch.cuda.get_device_name(0), "| python", sys.version.split()[0])
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
if not REPO.exists():
    assert sh(f"git clone -q https://{token}@github.com/cts0424/KD-ReID.git {REPO}") == 0, "clone failed (Internet on? GH_TOKEN secret?)"
sh(f"git -C {REPO} pull -q --ff-only && git -C {REPO} log --oneline -1")
assert sh(f"pip install -q -e {REPO}") == 0
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from kdreid.config import load_config

# ===== 1. Data: unzip if needed, then copy into the layout the loader expects =====
for z in IN.rglob("*.zip"):
    dst = RAW / z.stem
    if not dst.exists():
        print("unzipping", z.name, flush=True)
        zipfile.ZipFile(z).extractall(dst)

def find_split_dir(*keywords):
    hits = [p.parent for root in (IN, RAW) if root.exists() for p in root.rglob("bounding_box_train")
            if "__macosx" not in str(p).lower() and all(k in str(p).lower() for k in keywords)]
    assert hits, f"no bounding_box_train with {keywords} under /kaggle/input"
    return sorted(hits, key=lambda p: len(str(p)))[0]

def find_file(pattern):
    hits = sorted(IN.rglob(pattern))
    assert hits, f"{pattern} not found in the input dataset"
    return hits[0]

targets = {
    DATA / "Market-1501-v15.09.15": find_split_dir("market"),
    DATA / "cuhk03-np" / "detected": find_split_dir("detected"),
}
for dst, src in targets.items():
    if not dst.exists():
        print(f"copy {src} -> {dst}", flush=True)
        shutil.copytree(src, dst)

def overrides_for(cfg):
    """Point the Drive paths in a config at /kaggle; fail early if a needed file is missing."""
    ov = [f"data.root={DATA}", "data.num_workers=4"]
    if cfg.get("model", {}).get("pretrained_path"):
        ov.append(f"model.pretrained_path={find_file(cfg.model.backbone + '*.pth')}")
    teachers = cfg.get("teachers") or ([cfg.teacher] if cfg.get("teacher") else [])
    assert len(teachers) <= 1, "multi-teacher configs: pass the teacher paths in EXTRA"
    if teachers:
        t = teachers[0]
        assert str(t.get("backbone", "")).startswith("clipreid"), (
            "teacher is a trained run on Drive: attach its last.pth and pass teacher.checkpoint=... in EXTRA")
        ov.append(f"teacher.checkpoint={find_file('market_vit_clipreid*.pth')}")
    if cfg.get("unlabeled", {}).get("enabled"):
        ov.append(f"unlabeled.root={DATA}")
    return ov

# ===== 2. Train each run (resumes from OUT/last.pth if present), then evaluate =====
for run, seed in RUNS:
    cfg_path = Path("configs") / f"{run}.yaml"
    assert cfg_path.exists(), cfg_path
    cfg = load_config(cfg_path)
    out = WORK / "outputs" / run if seed == 42 else WORK / "outputs" / f"{run}_s{seed}"
    args = " ".join([f"--config {cfg_path}", f"seed={seed}", f"output_dir={out}", *overrides_for(cfg), EXTRA])
    t0 = time.time()
    rc = sh(f"python tools/train.py {args}")
    print(f"== {run} seed {seed}: exit {rc}, {(time.time() - t0) / 3600:.1f} h", flush=True)
    sh(f"grep -H Final {out}/log.txt")
    # Cross-domain (Market-trained models only): plain + per-domain BN statistics (--bn-recal)
    if rc == 0 and cfg.data.name == "market1501":
        sh(f"python tools/eval_matrix.py --runs {out} --bn-recal --datasets market1501:{DATA} cuhk03np:{DATA} "
           f"--out {WORK}/eval_matrix_{out.name}.json")